# v29 — Isu #2: Regime Awareness sbg Pengali Ketat Filter S/R (v28)

**Konteks (dari brief perbaikan Isu Signal v13, isu #2)**: robot v13 belum sadar rezim market
(ranging/sideways vs trending kuat). Krn v13 pada dasarnya flow-trend follower, dia berpotensi
lemah/rugi di kondisi ranging (false breakout/whipsaw).

**PENTING -- brief ini ditulis SEBELUM riset v20-v27 dilakukan.** Brief minta investigasi
"anomali Q2 ADX" dari v16 & deteksi regime H1/M15 dgn penyesuaian SL/TP -- KEDUANYA SUDAH
dicoba menyeluruh:
- **v25**: grid search 420 kombinasi ADX ceiling/exhaustion-by-ADX -- **0 kandidat robust**.
  Chi-square test (v27) membuktikan ADX bucket TIDAK signifikan (p=0.75) thd loss rate.
- **v20-v24**: regime-switching 4-kategori + BOS multi-timeframe + RSI divergence utk deteksi
  breakout/ranging -- >3500 kombinasi dicoba, SEMUA gagal (PF terbaik 0.32).

**Keputusan user**: JANGAN ulang riset regime-switching berdiri sendiri (sudah terbukti gagal).
Sebaliknya, gabungkan konsep regime awareness sbg **PENGALI KETAT** dari filter S/R yang SUDAH
TERBUKTI BEKERJA di v28 (PF full-period 0.87->1.25, unggul 6/8 tahun, diterapkan ke live
2026-09-01). Bukan filter regime berdiri sendiri, tapi modifier atas filter yang sudah ada.

**Hipotesis v29**: filter S/R v28 pakai threshold TETAP (`sr_near_atr_mult=3.0`,
`sr_strong_score_bonus=4.0`, `sr_min_atr_for_breakout=2.1`) di SEMUA kondisi. Mungkin threshold
yang optimal itu BEDA tergantung regime -- di kondisi Ranging (robot v13 lebih rawan whipsaw),
filter S/R mungkin perlu LEBIH KETAT (radius lebih besar, syarat lolos lebih sulit) drpd di
kondisi Trending Kuat (kondisi favorit v13, filter bisa lebih longgar spy tidak buang peluang).

**Deteksi regime**: ADX + ATR% dari H1 & M15 (pola v16 yg sudah tervalidasi cukup baik utk
klasifikasi 2x2 -- Trending>=25 vs Ranging<25, ATR% split median), BUKAN pendekatan baru yg
belum terbukti (BOS/RSI divergence dari v20-v24 sudah gagal, tidak diulang).

**Metodologi**: TRAIN (2019-2023)/TEST (2024-2026) walk-forward, robust lintas rezim (breakdown
per tahun), spread real 1.82, v12_score ASLI + OB filter + H1 alignment + filter S/R v28
sbg BASIS (bukan diganti). TIDAK ADA perubahan ke `usecase.py` sampai divalidasi & disetujui.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STRATEGY_NAME = "m5_scalping"
VERSION = "v29"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
(PROCESSED_DIR / VERSION).mkdir(parents=True, exist_ok=True)

INITIAL_EQUITY = 100.0
RISK_PCT = 0.01
CONTRACT_SIZE = 100.0
MIN_LOT = 0.01
LOT_STEP = 0.01
REAL_SPREAD = 1.82

MIN_SAMPLE_TRAIN = 30
MIN_SAMPLE_TEST = 15

pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load cache v28 (skor v12 + OB + H1 EMA + S/R H1/M15) + tambah ADX/ATR% dari H1 & M15

In [2]:
REGIME_CACHE_PATH = PROCESSED_DIR / VERSION / "df_2019_2026_regime.parquet"

if REGIME_CACHE_PATH.exists():
    print(f"Load dari cache: {REGIME_CACHE_PATH}")
    df = pd.read_parquet(REGIME_CACHE_PATH)
else:
    print("Belum ada cache -- load v28 base + merge ADX/ATR H1 & M15 utk regime...")
    v28_cache = PROCESSED_DIR / "v28" / "df_2019_2026_sr.parquet"
    assert v28_cache.exists(), "Cache v28 belum ada"
    df = pd.read_parquet(v28_cache)

    for tf_file, tf_delta, prefix in [("h1", pd.Timedelta(hours=1), "h1"), ("m15", pd.Timedelta(minutes=15), "m15")]:
        df_tf = pd.read_csv(
            PROCESSED_DIR / "v01" / f"xauusd_{tf_file}_full_indicators.csv",
            usecols=["datetime", "close", "adx", "atr"],
        )
        df_tf["datetime"] = pd.to_datetime(df_tf["datetime"])
        df_tf = df_tf.sort_values("datetime").reset_index(drop=True)
        df_tf["atr_pct"] = df_tf["atr"] / df_tf["close"] * 100
        df_tf["available_at"] = df_tf["datetime"] + tf_delta
        df_tf = df_tf.rename(columns={"adx": f"{prefix}_regime_adx", "atr_pct": f"{prefix}_regime_atr_pct"})
        cols = ["available_at", f"{prefix}_regime_adx", f"{prefix}_regime_atr_pct"]
        df = pd.merge_asof(
            df.sort_values("datetime"), df_tf[cols].sort_values("available_at"),
            left_on="datetime", right_on="available_at", direction="backward",
        )
        df = df.drop(columns=["available_at"])
        print(f"  merged regime cols {prefix.upper()}")

    df.to_parquet(REGIME_CACHE_PATH, index=False)
    print(f"Tersimpan ke cache: {REGIME_CACHE_PATH}")

print(f"\nTotal candle: {len(df)}, {df['datetime'].min()} -> {df['datetime'].max()}")
print(f"Kolom regime: {[c for c in df.columns if 'regime' in c]}")

Belum ada cache -- load v28 base + merge ADX/ATR H1 & M15 utk regime...


  merged regime cols H1


  merged regime cols M15


Tersimpan ke cache: D:\Projects\robot-scalping\dataset\processed\m5_scalping\v29\df_2019_2026_regime.parquet

Total candle: 518403, 2019-01-01 23:00:00+00:00 -> 2026-08-06 12:35:00+00:00
Kolom regime: ['h1_regime_adx', 'h1_regime_atr_pct', 'm15_regime_adx', 'm15_regime_atr_pct']


## 2. Klasifikasi regime 2x2 (pola v16 yang sudah tervalidasi): Trending vs Ranging x ADX H1/M15

**Kenapa pola v16, bukan pendekatan baru**: v16 sudah membuktikan klasifikasi ADX>=25
(Trending) vs <25 (Ranging) itu punya breakdown performa yg BERBEDA scr deskriptif (walau
semua kombinasi tetap profitable). v20-v24 (BOS/RSI utk deteksi regime scr lebih canggih) SEMUA
gagal -- jadi v29 kembali ke definisi regime yang PALING SEDERHANA & sudah terbukti berguna scr
deskriptif, bukan reka ulang deteksi regime dari nol.

In [3]:
REGIME_ADX_THRESHOLD = 25.0

def classify_regime(h1_adx, m15_adx):
    """Ranging kalau KEDUA h1 & m15 ADX < threshold (whipsaw risk tinggi di kedua timeframe);
    Trending kalau SALAH SATU >= threshold (ada momentum di minimal 1 timeframe konteks)."""
    if pd.isna(h1_adx) or pd.isna(m15_adx):
        return "UNKNOWN"
    if h1_adx < REGIME_ADX_THRESHOLD and m15_adx < REGIME_ADX_THRESHOLD:
        return "RANGING"
    return "TRENDING"

df["regime"] = df.apply(lambda r: classify_regime(r["h1_regime_adx"], r["m15_regime_adx"]), axis=1)
print("=== Distribusi regime (semua candle 2019-2026) ===")
print(df["regime"].value_counts())
print()
print((df["regime"].value_counts(normalize=True) * 100).round(1))

=== Distribusi regime (semua candle 2019-2026) ===
regime
TRENDING    348940
RANGING     169451
UNKNOWN         12
Name: count, dtype: int64

regime
TRENDING    67.3
RANGING     32.7
UNKNOWN      0.0
Name: proportion, dtype: float64


## 3. Backtest engine v29: v13 + filter S/R (basis v28) + PENGALI ketat per regime

In [4]:
def check_h1_alignment_v29(h1_ema_50, h1_ema_200, direction: str) -> bool:
    if h1_ema_50 is None or h1_ema_200 is None or not np.isfinite(h1_ema_50) or not np.isfinite(h1_ema_200):
        return True
    h1_trend = "UP" if h1_ema_50 > h1_ema_200 else ("DOWN" if h1_ema_50 < h1_ema_200 else "FLAT")
    if direction == "BUY" and h1_trend == "DOWN":
        return False
    if direction == "SELL" and h1_trend == "UP":
        return False
    return True


def run_backtest_v29(
    df_signals: pd.DataFrame,
    adx_min: float = 18.0,
    min_signal_score: float = 9.0,
    sl_mult: float = 2.0,
    tp_mult: float = 4.0,
    max_hold: int = 12,
    # Filter S/R basis v28 (default = parameter terbaik v28 yang SUDAH live)
    sr_near_atr_mult_trending: float = 3.0,
    sr_near_atr_mult_ranging: float = 3.0,   # kalau != trending, itu "pengali" ketat
    sr_strong_score_bonus_trending: float = 4.0,
    sr_strong_score_bonus_ranging: float = 4.0,
    sr_min_atr_for_breakout_trending: float = 2.1,
    sr_min_atr_for_breakout_ranging: float = 2.1,
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
    use_fixed_lot: bool = False,
    fixed_lot_value: float = MIN_LOT,
    initial_equity_override: float = None,
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    h1_res_arr = df_signals["h1_sr_resistance"].to_numpy()
    h1_sup_arr = df_signals["h1_sr_support"].to_numpy()
    m15_res_arr = df_signals["m15_sr_resistance"].to_numpy()
    m15_sup_arr = df_signals["m15_sr_support"].to_numpy()
    regime_arr = df_signals["regime"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    trades = []
    base_equity = initial_equity_override if initial_equity_override is not None else INITIAL_EQUITY
    equity = base_equity
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue
        if adx < adx_min:
            i += 1
            continue

        direction = None
        if score >= min_signal_score:
            direction = "BUY"
        elif score <= -min_signal_score:
            direction = "SELL"
        if direction is None:
            i += 1
            continue

        if require_ob_filter:
            opposing_ob = (
                (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
            )
            if opposing_ob:
                i += 1
                continue
        if require_h1_alignment:
            if not check_h1_alignment_v29(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                i += 1
                continue

        # --- Filter S/R, threshold TERGANTUNG regime saat ini ---
        is_ranging = regime_arr[i] == "RANGING"
        near_atr_mult = sr_near_atr_mult_ranging if is_ranging else sr_near_atr_mult_trending
        strong_score_bonus = sr_strong_score_bonus_ranging if is_ranging else sr_strong_score_bonus_trending
        min_atr_for_breakout = sr_min_atr_for_breakout_ranging if is_ranging else sr_min_atr_for_breakout_trending

        if direction == "BUY":
            candidates = [v for v in (h1_res_arr[i], m15_res_arr[i]) if np.isfinite(v)]
            opposing_level = min(candidates) if candidates else None
        else:
            candidates = [v for v in (h1_sup_arr[i], m15_sup_arr[i]) if np.isfinite(v)]
            opposing_level = max(candidates) if candidates else None

        if opposing_level is not None:
            dist = abs(opposing_level - close)
            is_near = dist <= (near_atr_mult * atr)
            is_strong_signal = abs(score) >= (min_signal_score + strong_score_bonus)
            is_breakout_atr = atr >= min_atr_for_breakout
            if is_near and not is_strong_signal and not is_breakout_atr:
                i += 1
                continue

        sl_points = sl_mult * atr
        tp_points = tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        if use_fixed_lot:
            lot = fixed_lot_value
        else:
            risk_amount = equity * RISK_PCT
            lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 and risk_amount > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        trades.append({
            "entry_time": entry_time, "direction": direction, "regime": regime_arr[i], "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)


def evaluate(trades: pd.DataFrame, initial_equity: float) -> dict:
    if trades.empty:
        return {"total_trades": 0, "win_rate_pct": 0, "profit_factor": 0, "net_pnl": 0, "max_drawdown_pct": 0}
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = losses["pnl"].sum()
    equity_series = pd.Series([initial_equity] + trades["equity_after"].tolist())
    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max * 100
    return {
        "total_trades": len(trades),
        "win_rate_pct": round(len(wins) / len(trades) * 100, 2),
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "net_pnl": round(gross_profit + gross_loss, 2),
        "max_drawdown_pct": round(drawdown.min(), 2),
    }

print("Backtest engine v29 siap.")

Backtest engine v29 siap.


## 4. TRAIN/TEST split & Baseline (v28 murni -- filter S/R TANPA pengali regime, sr sama di semua kondisi)

In [5]:
TRAIN_END = pd.Timestamp("2024-01-01", tz="UTC")
df_train = df[df["datetime"] < TRAIN_END].reset_index(drop=True)
df_test = df[df["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"TRAIN (2019-2023): {len(df_train)} candle | TEST (2024-2026): {len(df_test)} candle")

trades_base_train = run_backtest_v29(df_train)  # default = v28 params, sama di semua regime
trades_base_test = run_backtest_v29(df_test)
baseline_train = evaluate(trades_base_train, INITIAL_EQUITY)
baseline_test = evaluate(trades_base_test, INITIAL_EQUITY)
print("=== Baseline: v28 murni (S/R filter SAMA di semua regime, spt yg sudah live) ===")
print("TRAIN:", baseline_train)
print("TEST :", baseline_test)

TRAIN (2019-2023): 351136 candle | TEST (2024-2026): 167267 candle


=== Baseline: v28 murni (S/R filter SAMA di semua regime, spt yg sudah live) ===
TRAIN: {'total_trades': 944, 'win_rate_pct': 23.2, 'profit_factor': np.float64(0.6), 'net_pnl': np.float64(-674.81), 'max_drawdown_pct': np.float64(-674.81)}
TEST : {'total_trades': 810, 'win_rate_pct': 50.12, 'profit_factor': np.float64(1.76), 'net_pnl': np.float64(1741.88), 'max_drawdown_pct': np.float64(-102.96)}


## 4b. Cek pola dulu: apakah v28 (S/R filter tetap) memang lebih lemah saat RANGING?

Sebelum grid search pengali regime, verifikasi dulu hipotesis dasarnya -- apakah trade v28
yang lolos filter S/R (yang sudah ada) itu MEMANG lebih sering rugi saat regime RANGING drpd
TRENDING. Kalau tidak ada beda, menambah pengali regime kemungkinan tidak akan membantu.

In [6]:
print("=== v28 (baseline): performa per regime, TRAIN ===")
for regime, g in trades_base_train.groupby("regime"):
    m = evaluate(g, INITIAL_EQUITY)
    print(f"  {regime}: n={m['total_trades']}, win_rate={m['win_rate_pct']}%, PF={m['profit_factor']}, net_pnl=${m['net_pnl']}")

print("\n=== v28 (baseline): performa per regime, TEST ===")
for regime, g in trades_base_test.groupby("regime"):
    m = evaluate(g, INITIAL_EQUITY)
    print(f"  {regime}: n={m['total_trades']}, win_rate={m['win_rate_pct']}%, PF={m['profit_factor']}, net_pnl=${m['net_pnl']}")

=== v28 (baseline): performa per regime, TRAIN ===
  RANGING: n=180, win_rate=22.22%, PF=0.68, net_pnl=$-96.28
  TRENDING: n=764, win_rate=23.43%, PF=0.58, net_pnl=$-578.54

=== v28 (baseline): performa per regime, TEST ===
  RANGING: n=193, win_rate=50.26%, PF=1.64, net_pnl=$340.87
  TRENDING: n=617, win_rate=50.08%, PF=1.79, net_pnl=$1401.02


## 5. Grid search: pengali ketat filter S/R KHUSUS saat regime RANGING

Trending TETAP pakai parameter v28 yang sudah live (near_atr_mult=3.0, strong_score_bonus=4.0,
min_atr_for_breakout=2.1) -- TIDAK diubah, krn itu kondisi favorit v13 yang sudah tervalidasi.
Cuma parameter versi RANGING yang digrid search, dicoba LEBIH KETAT dari versi trending.

In [7]:
import itertools
import time as _time

FIXED_TRENDING = dict(
    sr_near_atr_mult_trending=3.0, sr_strong_score_bonus_trending=4.0, sr_min_atr_for_breakout_trending=2.1,
)

GRID = {
    "sr_near_atr_mult_ranging": [3.0, 4.0, 5.0, 6.0],       # >= trending punya (lebih ketat/radius lebih besar)
    "sr_strong_score_bonus_ranging": [4.0, 6.0, 8.0],        # >= trending (butuh skor lebih kuat lagi)
    "sr_min_atr_for_breakout_ranging": [2.1, 2.5, 3.0, 3.5], # >= trending (butuh ATR lebih 'bertenaga')
}

combos = list(itertools.product(*GRID.values()))
print(f"Total kombinasi grid: {len(combos)}")

t0 = _time.time()
grid_results = []
for idx, combo in enumerate(combos):
    params = dict(zip(GRID.keys(), combo))
    trades = run_backtest_v29(df_train, **FIXED_TRENDING, **params)
    metrics = evaluate(trades, INITIAL_EQUITY)
    metrics.update(params)
    grid_results.append(metrics)
    if (idx + 1) % 15 == 0:
        print(f"  [{idx+1}/{len(combos)}] {_time.time()-t0:.0f}s")

grid_df = pd.DataFrame(grid_results)
print(f"\nGrid search selesai dalam {_time.time()-t0:.0f}s")

grid_valid = grid_df[grid_df["total_trades"] >= MIN_SAMPLE_TRAIN].sort_values("profit_factor", ascending=False)
print(f"\n=== Top 15 kandidat (sample TRAIN >= {MIN_SAMPLE_TRAIN}) ===")
print(grid_valid.head(15).to_string(index=False))

beating = grid_valid[grid_valid["profit_factor"] > baseline_train["profit_factor"]]
print(f"\nBaseline (v28) TRAIN PF: {baseline_train['profit_factor']}")
print(f"Kandidat mengungguli baseline v28 TRAIN: {len(beating)} dari {len(grid_valid)}")

Total kombinasi grid: 48


  [15/48] 13s


  [30/48] 26s


  [45/48] 39s



Grid search selesai dalam 42s

=== Top 15 kandidat (sample TRAIN >= 30) ===
 total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct  sr_near_atr_mult_ranging  sr_strong_score_bonus_ranging  sr_min_atr_for_breakout_ranging
          871         24.11           0.63  -591.29           -591.29                       5.0                            8.0                              2.1
          856         24.30           0.63  -584.02           -584.02                       6.0                            4.0                              2.1
          855         24.33           0.63  -577.41           -577.41                       6.0                            8.0                              2.1
          855         24.33           0.63  -577.41           -577.41                       6.0                            6.0                              2.1
          871         24.11           0.63  -591.29           -591.29                       5.0                            

## 6. Validasi TEST out-of-sample (kandidat yang mengungguli baseline v28 di TRAIN)

In [8]:
candidates_passing = beating.head(15)
print(f"Kandidat TRAIN mengungguli baseline v28: {len(candidates_passing)}")

if len(candidates_passing) == 0:
    print("\n>>> TIDAK ADA kandidat mengungguli baseline v28 di TRAIN. Validasi TEST DIBATALKAN.")
else:
    test_results = []
    for _, row in candidates_passing.iterrows():
        params = {k: row[k] for k in GRID.keys()}
        trades_test = run_backtest_v29(df_test, **FIXED_TRENDING, **params)
        m_test = evaluate(trades_test, INITIAL_EQUITY)
        test_results.append({**params, "train_pf": row["profit_factor"], "train_n": row["total_trades"],
                              "test_pf": m_test["profit_factor"], "test_n": m_test["total_trades"],
                              "test_wr": m_test["win_rate_pct"], "test_netpnl": m_test["net_pnl"],
                              "test_maxdd": m_test["max_drawdown_pct"]})

    test_df = pd.DataFrame(test_results)
    print("\n=== Validasi TEST utk kandidat yang menang di TRAIN ===")
    print(test_df.to_string(index=False))

    print(f"\nBaseline (v28) TEST: PF={baseline_test['profit_factor']}, net_pnl={baseline_test['net_pnl']}, n={baseline_test['total_trades']}")

    robust = test_df[(test_df["test_pf"] > baseline_test["profit_factor"]) & (test_df["test_n"] >= MIN_SAMPLE_TEST)]
    print(f"\n>>> Kandidat ROBUST (unggul TRAIN & TEST vs baseline v28, sample TEST>={MIN_SAMPLE_TEST}): {len(robust)}")
    if len(robust) > 0:
        print(robust.to_string(index=False))

Kandidat TRAIN mengungguli baseline v28: 15



=== Validasi TEST utk kandidat yang menang di TRAIN ===
 sr_near_atr_mult_ranging  sr_strong_score_bonus_ranging  sr_min_atr_for_breakout_ranging  train_pf  train_n  test_pf  test_n  test_wr  test_netpnl  test_maxdd
                      5.0                            8.0                              2.1      0.63    871.0     1.78     796    50.63      1776.70      -87.29
                      6.0                            4.0                              2.1      0.63    856.0     1.78     795    50.69      1778.70      -86.10
                      6.0                            8.0                              2.1      0.63    855.0     1.78     795    50.69      1778.70      -86.10
                      6.0                            6.0                              2.1      0.63    855.0     1.78     795    50.69      1778.70      -86.10
                      5.0                            6.0                              2.1      0.63    871.0     1.78     796    50.63      177

## 7. Full-period 2019-2026 comparison (fixed lot, no kill-switch -- basis sama spt v28 Section 9)

Kalau ada kandidat robust, bandingkan full-period apple-to-apple dgn v28 (yang sudah live)
pakai basis yang PERSIS SAMA (lot fixed 0.03, modal $2000, tanpa kill-switch).

In [9]:
if 'robust' in dir() and len(robust) > 0:
    best = robust.sort_values("test_pf", ascending=False).iloc[0]
    best_params = {k: best[k] for k in GRID.keys()}
    print(f"Kandidat terbaik: {best_params}")

    common_kwargs = dict(use_fixed_lot=True, fixed_lot_value=0.03, initial_equity_override=2000.0)
    trades_v28_full = run_backtest_v29(df, **common_kwargs)  # v28 params default (sama di semua regime)
    trades_v29_full = run_backtest_v29(df, **FIXED_TRENDING, **best_params, **common_kwargs)

    m_v28 = evaluate(trades_v28_full, 2000.0)
    m_v29 = evaluate(trades_v29_full, 2000.0)

    print("\n=== FULL PERIOD 2019-2026: v28 (live sekarang) vs v29 (S/R + pengali regime) ===")
    compare_df = pd.DataFrame([
        {"strategy": "v28 (live)", **m_v28},
        {"strategy": "v29 (regime-aware S/R)", **m_v29},
    ])
    print(compare_df.to_string(index=False))

    # Breakdown per tahun
    def yearly_breakdown(trades, label):
        trades = trades.copy()
        trades["year"] = pd.to_datetime(trades["entry_time"]).dt.year
        rows = []
        for year, g in trades.groupby("year"):
            m = evaluate(g, 2000.0)
            rows.append({"year": year, "strategy": label, **m})
        return pd.DataFrame(rows)

    yearly_v28 = yearly_breakdown(trades_v28_full, "v28")
    yearly_v29 = yearly_breakdown(trades_v29_full, "v29")
    yearly_compare = pd.concat([yearly_v28, yearly_v29]).sort_values(["year", "strategy"])
    print("\n=== Breakdown per tahun ===")
    print(yearly_compare[["year", "strategy", "total_trades", "win_rate_pct", "profit_factor", "net_pnl"]].to_string(index=False))

    years_v29_better = sum(
        1 for year in sorted(yearly_v28["year"].unique())
        if len(yearly_v29[yearly_v29["year"]==year]) and len(yearly_v28[yearly_v28["year"]==year])
        and yearly_v29[yearly_v29["year"]==year]["profit_factor"].values[0] > yearly_v28[yearly_v28["year"]==year]["profit_factor"].values[0]
    )
    print(f"\nv29 mengungguli v28 di {years_v29_better} dari {len(yearly_v28)} tahun")
else:
    print("Tidak ada kandidat robust dari Section 6 -- lewati full-period comparison.")

Kandidat terbaik: {'sr_near_atr_mult_ranging': np.float64(6.0), 'sr_strong_score_bonus_ranging': np.float64(6.0), 'sr_min_atr_for_breakout_ranging': np.float64(2.5)}



=== FULL PERIOD 2019-2026: v28 (live sekarang) vs v29 (S/R + pengali regime) ===
              strategy  total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct
            v28 (live)          1754         35.63           1.25  2967.73           -116.84
v29 (regime-aware S/R)          1590         36.92           1.31  3336.49            -98.33

=== Breakdown per tahun ===
 year strategy  total_trades  win_rate_pct  profit_factor  net_pnl
 2019      v28           176          5.11           0.08  -724.25
 2019      v29           155          5.16           0.08  -660.63
 2020      v28           219         31.96           0.69  -438.09
 2020      v29           197         34.52           0.74  -317.06
 2021      v28           169         23.08           0.72  -251.27
 2021      v29           151         23.84           0.74  -208.59
 2022      v28           172         30.23           0.75  -236.09
 2022      v29           155         30.97           0.78  -189.45
 2023  

## 8. Kesimpulan

*(diisi setelah lihat hasil eksekusi lengkap Section 3-7 -- placeholder)*